In [1]:
import pandas as pd
import numpy as np

columns = ['engine_id','cycle','op1','op2','op3'] + [f'sensor{i}' for i in range(1,22)]

train = pd.read_csv(
    'train_FD001.txt',
    sep=' ',
    header=None
)

train = train.iloc[:,:26]

train.columns = columns

train.head()

,engine_id,cycle,op1,op2,op3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388.0,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388.0,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388.0,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388.0,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388.0,100.0,38.90,23.4044


In [2]:
print(train.shape)

train.info()

(12315, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12315 entries, 0 to 12314
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   engine_id  12315 non-null  int64  
 1   cycle      12315 non-null  int64  
 2   op1        12315 non-null  float64
 3   op2        12315 non-null  float64
 4   op3        12315 non-null  float64
 5   sensor1    12315 non-null  float64
 6   sensor2    12315 non-null  float64
 7   sensor3    12315 non-null  float64
 8   sensor4    12315 non-null  float64
 9   sensor5    12315 non-null  float64
 10  sensor6    12315 non-null  float64
 11  sensor7    12315 non-null  float64
 12  sensor8    12315 non-null  float64
 13  sensor9    12315 non-null  float64
 14  sensor10   12315 non-null  float64
 15  sensor11   12315 non-null  float64
 16  sensor12   12315 non-null  float64
 17  sensor13   12315 non-null  float64
 18  sensor14   12315 non-null  float64
 19  sensor15   12315 non-null  float64

In [3]:
max_cycle = train.groupby('engine_id')['cycle'].max()

train = train.merge(
    max_cycle.rename('max_cycle'),
    on='engine_id'
)

train['RUL'] = train['max_cycle'] - train['cycle']

train.head()

,engine_id,cycle,op1,op2,op3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21,max_cycle,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8138.62,8.4195,0.03,392,2388.0,100.0,39.06,23.4190,192,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8131.49,8.4318,0.03,392,2388.0,100.0,39.00,23.4236,192,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8133.23,8.4178,0.03,390,2388.0,100.0,38.95,23.3442,192,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8133.83,8.3682,0.03,392,2388.0,100.0,38.88,23.3739,192,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8133.80,8.4294,0.03,393,2388.0,100.0,38.90,23.4044,192,187


In [4]:
features = train.drop(
    ['engine_id','cycle','max_cycle','RUL'],
    axis=1
)

target = train['RUL']

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [6]:
from sklearn.metrics import mean_absolute_error

pred = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    pred
)

print("MAE:", mae)

MAE: 23.88274868047097


In [7]:
results = pd.DataFrame({
    'Actual_RUL': y_test,
    'Predicted_RUL': pred
})

results.head()

,Actual_RUL,Predicted_RUL
5640,152,109.59
8720,74,114.06
1022,93,75.88
8774,20,34.74
9992,129,129.41


In [8]:
results.to_csv("rul_predictions.csv", index=False)

In [9]:
import joblib

joblib.dump(model, "rul_model.pkl")

['rul_model.pkl']

In [10]:
import os

print(os.listdir())

['.config', 'test_FD001.txt', 'rul_predictions.csv', 'train_FD004.txt', 'train_FD003.txt', 'train_FD002.txt', 'rul_model.pkl', 'train_FD001.txt', 'RUL_FD001.txt', 'sample_data']


In [11]:
from google.colab import files

files.download("rul_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import os

size = os.path.getsize("rul_model.pkl")/(1024*1024)

print("Size:", size, "MB")

Size: 79.23768711090088 MB
